# 🌟 Diabetes Progression Prediction: A Beginner's Guide

Welcome! In this notebook, we will build a **Multiple Linear Regression** model from scratch using Python and NumPy. 

Our goal is to predict the progression of diabetes in patients based on various health metrics. We will break down each part of the code to understand how it works.

## 1. Import Libraries

First, we need to import the necessary libraries.
*   **NumPy (`np`)**: Used for efficient numerical operations and matrix math.
*   **Pandas (`pd`)**: Used for loading and handling the dataset.
*   **Matplotlib (`plt`)**: Used for creating visualizations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the Dataset

We load our data from a CSV file using Pandas. 

*   `X` represents our **features** (inputs like age, BMI, blood pressure, etc.). We assume the target variable is labeled 'Y', so we drop it to get just the features.
*   `y` is our **target** variable (disease progression) that we want to predict. We convert it to a NumPy array to make it compatible with our math operations.

In [ ]:
# Load dataset
df = pd.read_csv('assets/diabetes.tab.csv')

X = df.drop('Y', axis=1).values  # Prepare features (Drop 'Y' column)
y = df[['Y']].to_numpy()         # Prepare target (Select 'Y' column)

# Check the shape of our data
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

## 3. Data Normalization

Machine learning models often work better when all features are on a similar scale. We perform **Normalization** to bring all values to a common range.

We use the formula:
$$ X_{scaled} = \frac{X - X_{mean}}{X_{max} - X_{min}} $$

This centers the data around 0 and scales it based on the range of values.

In [ ]:
# Normalization
X_mean = X.mean(axis=0)                      # Calculate mean of each column
X_range = X.max(axis=0) - X.min(axis=0)      # Calculate range (max - min) of each column
X_range[X_range == 0] = 1                    # Prevent division by zero

X_scaled = (X - X_mean) / X_range            # Apply normalization

## 4. Add Intercept Term

In the linear equation $y = \theta_0 + \theta_1 x_1 + ...$, there is a constant term $\theta_0$ (the intercept).

To handle this using matrix multiplication, we add a column of **ones** to the start of our feature matrix `X`. This allows $\theta_0$ to be multiplied by 1.

In [ ]:
# Add intercept term
# Create a column of ones with the same number of rows as X
ones_column = np.ones((X_scaled.shape[0], 1))

# Stack the ones column horizontally with X
X_scaled = np.hstack([ones_column, X_scaled])

print(f"New shape of X after adding intercept: {X_scaled.shape}")

## 5. Set Hyperparameters

Before training, we need to set some configuration variables, known as **hyperparameters**:

*   **Alpha ($\alpha$)**: The **Learning Rate**. It controls how big of a step we take when updating our model. If it's too big, we might overshoot; too small, and training takes forever.
*   **Iterations**: How many times the model will look at the dataset to learn.
*   **m**: The number of training examples (rows in our data).

In [ ]:
# Hyperparameters (optimized)
alpha = 0.01        # Learning rate
iterations = 2000   # Number of training loops
m = len(y)          # Number of samples

## 6. Initialize Parameters (Theta)

We need a place to store the weights (coefficients) that our model learns. These are denoted by Theta ($\theta$).

We start by initializing them to **zero**. As the model trains, these values will be updated to better predict the target.

In [ ]:
# Initialize theta with zeros
# The size is (number of columns in X, 1)
theta = np.zeros((X_scaled.shape[1], 1))

## 7. Gradient Descent Training Loop

This is the core of the learning process. For each iteration, we do the following:

1.  **Make Predictions**: Calculate $X \cdot \theta$.
2.  **Calculate Error**: Find the difference between Predictions and Actual values ($y$).
3.  **Compute Gradient**: Find the direction to move $\theta$ to reduce error.
4.  **Update Theta**: Adjust $\theta$ using the learning rate and gradient.
5.  **Track Cost**: Calculate the Mean Squared Error (MSE) to see if the model is improving.

In [ ]:
# Gradient Descent
cost_history = []

for i in range(iterations):
    # 1. Predictions: Matrix multiplication of X and theta
    predictions = X_scaled @ theta
    
    # 2. Error: Difference between predicted and actual values
    error = predictions - y
    
    # 3. Gradient Calculation
    gradient = (1/m) * X_scaled.T @ error
    
    # 4. Update Theta
    theta -= alpha * gradient
    
    # 5. Calculate Cost (Mean Squared Error)
    cost = (1/(2*m)) * np.sum(error**2)
    cost_history.append(cost)

## 8. Final Results and Visualization

Now that training is complete, let's look at the results.

*   We print the optimized values of $\theta$.
*   We plot variables to visualize how well our predictions match the actual disease progression.Ideally, the points should fall along the diagonal dashed line.

In [ ]:
# Final predictions
y_pred = X_scaled @ theta

print("Optimized Parameters:")
print(f"Learning rate (alpha): {alpha}")
print(f"Iterations: {iterations}")
print(f"\nFinal Coefficients (theta):\n{theta.ravel()}")

# Actual vs Predicted plot
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5, color='blue', label='Predictions')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2, label='Perfect Prediction')

plt.xlabel("Actual Disease Progression")
plt.ylabel("Predicted Disease Progression")
plt.title("Full Dataset Performance: Actual vs Predicted")
plt.legend()
plt.grid(True)
plt.show()